# Amharic ASR Training - Kaggle GPU

Use GPU accelerator: **T4** or better

Dataset: `Harbidel/amharic-asr-merged` (74,691 clips, 243 hours)

In [ ]:
!pip install -q datasets soundfile

In [ ]:
import os, torch, json, time
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from datasets import load_dataset

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
CONFIG = {
    "vocab_size": 1024,
    "sample_rate": 16000,
    "encoder_dim": 256,
    "encoder_layers": 6,
    "attention_heads": 4,
    "ffn_dim": 1024,
    "batch_size": 16,
    "learning_rate": 2e-3,
    "max_epochs": 30,
    "early_stopping_patience": 5,
    "dataset_name": "Harbidel/amharic-asr-merged",
    "output_dir": "./checkpoints"
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))

In [ ]:
print("Loading dataset...")
dataset = load_dataset(CONFIG["dataset_name"])
print(f"Train: {len(dataset['train'])} | Val: {len(dataset['validation'])} | Test: {len(dataset['test'])}")
print(f"Sample: {dataset['train'][0]['text'][:80]}...")

In [ ]:
class AmharicASRDataset(Dataset):
    def __init__(self, ds, sr=16000):
        self.ds = ds
        self.sr = sr
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        s = self.ds[idx]
        audio = np.array(s["audio"]["array"], dtype=np.float32)
        orig_sr = s["audio"]["sampling_rate"]
        if orig_sr != self.sr:
            from scipy.signal import resample
            audio = resample(audio, int(len(audio) / orig_sr * self.sr))
        if np.max(np.abs(audio)) > 0:
            audio = audio / np.max(np.abs(audio))
        return {"audio": torch.FloatTensor(audio), "text": s["text"]}

In [ ]:
class CharTokenizer:
    def __init__(self, vocab_size=1024):
        self.c2i = {"<blank>": 0, "<unk>": 1}
        chars = list("ሀለሐመሠረሰሸቀበተቸኀነአከወዘየደደጀገገጠጰጸፀፈፐאבగდვზთიკლმნოპჟრსტუფქღყშჩცძწჭხჯჰ")
        chars += list("0123456789.,!?;:\" ")
        for i, c in enumerate(chars[:vocab_size-2]):
            self.c2i[c] = i + 2
        self.i2i = {v:k for k,v in self.c2i.items()}
    def encode(self, text):
        return [self.c2i.get(c, 1) for c in text]
    def decode(self, ids):
        return "".join([self.i2i.get(i, "") for i in ids if i > 0])

In [ ]:
class AudioPrep(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64, 64, stride=2, padding=32), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 32, stride=3, padding=16), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 16, stride=2, padding=8), nn.BatchNorm1d(256), nn.ReLU()
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1)).transpose(1, 2)

class ASRModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.prep = AudioPrep()
        layer = nn.TransformerEncoderLayer(256, 4, 1024, 0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, 6)
        self.head = nn.Linear(256, cfg["vocab_size"])
    def forward(self, x):
        x = self.prep(x)
        x = self.encoder(x)
        return self.head(x)

In [ ]:
def train():
    device = torch.device("cuda")
    tok = CharTokenizer(CONFIG["vocab_size"])
    model = ASRModel(CONFIG).to(device)
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
    
    train_ds = AmharicASRDataset(dataset["train"])
    val_ds = AmharicASRDataset(dataset["validation"])
    train_dl = DataLoader(train_ds, 16, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(val_ds, 16, shuffle=False, num_workers=2)
    
    optimizer = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=0.01)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    
    best_loss = float("inf")
    patience = 0
    
    for epoch in range(CONFIG["max_epochs"]):
        model.train()
        total_loss = 0
        
        for i, batch in enumerate(train_dl):
            audio = batch["audio"].to(device)
            logits = model(audio)
            
            targets = [tok.encode(t) for t in batch["text"]]
            t_lens = [len(t) for t in targets]
            max_t = max(t_lens)
            padded = torch.zeros(len(targets), max_t, dtype=torch.long)
            for j, t in enumerate(targets):
                padded[j, :len(t)] = torch.tensor(t)
            
            log_probs = torch.log_softmax(logits, dim=-1)
            input_lens = torch.full((audio.size(0),), logits.size(1), dtype=torch.long)
            loss = criterion(log_probs.permute(1,0,2), padded, input_lens, torch.tensor(t_lens))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()
            
            if i % 100 == 0:
                print(f"  [{epoch}] Batch {i}/{len(train_dl)} Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(train_dl)
        print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience = 0
            torch.save(model.state_dict(), f"{CONFIG['output_dir']}/best.pt")
            print(f"  Saved best model (loss={avg_loss:.4f})")
        else:
            patience += 1
            if patience >= CONFIG["early_stopping_patience"]:
                print("Early stopping!")
                break
        
        if (epoch+1) % 5 == 0:
            torch.save({"epoch": epoch, "model": model.state_dict(), "config": CONFIG}, 
                       f"{CONFIG['output_dir']}/epoch_{epoch+1}.pt")
    
    torch.save(model.state_dict(), f"{CONFIG['output_dir']}/final.pt")
    print("Training complete!")

In [ ]:
train()